<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/PanCancer_Classification_Survival_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Pan-Cancer Classification & Survival Analysis (Real TCGA Data)

# Project 11 — Advanced Bioinformatics

Yeh ek **end-to-end, publication-style pan-cancer genomics analysis** hai jo **real TCGA (The Cancer Genome Atlas)** data — cBioPortal ke public REST API se — use karta hai. Is notebook mein hum multiple cancer types ka gene expression data integrate kar ke:

1. Cancer type **classify** karain gay expression profile se
2. **Survival analysis** karain gay (Kaplan-Meier curves, log-rank tests)
3. **Cox Proportional Hazards regression** se prognostic genes identify karain gay
4. Patient-level **risk stratification** karain gay
5. Aakhir mein ek **runtime prediction tool** banayein gay

---

## 📋 Table of Contents

| Section | Content |
|---|---|
| 1 | Setup & Installation |
| 2 | Data Acquisition — cBioPortal API (real TCGA studies) |
| 3 | Data Integration & Cleaning |
| 4 | Exploratory Data Analysis (Plotly) |
| 5 | Pan-Cancer Type Classification (ML) |
| 6 | Model Evaluation — Confusion Matrix, ROC (One-vs-Rest) |
| 7 | Kaplan-Meier Survival Analysis |
| 8 | Gene-Stratified Survival (High vs Low Expression) |
| 9 | Cox Proportional Hazards Regression |
| 10 | Multivariate Risk Score & Stratification |
| 11 | 🧬 **Runtime Prediction** — Apna Patient Data Direct Input Karein |




## 1. Setup & Installation




In [1]:
!pip install -q lifelines plotly scikit-learn pandas numpy requests ipywidgets

import numpy as np
import pandas as pd
import requests
import time
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                              confusion_matrix, roc_curve, f1_score)

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)
BASE_URL = "https://www.cbioportal.org/api"


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.4 MB/s eta 0:00:00


## 2. Data Acquisition — Real TCGA Data (cBioPortal API)

Hum **5 TCGA PanCancer Atlas studies** ka real clinical aur mRNA expression (Z-score) data fetch kar rahe hain. cBioPortal ek free, public API hai — koi authentication/API key nahi chahiye.




In [2]:
STUDIES = {
    "brca_tcga_pan_can_atlas_2018": "Breast Cancer (BRCA)",
    "luad_tcga_pan_can_atlas_2018": "Lung Adenocarcinoma (LUAD)",
    "coadread_tcga_pan_can_atlas_2018": "Colorectal Cancer (COADREAD)",
    "prad_tcga_pan_can_atlas_2018": "Prostate Cancer (PRAD)",
    "kirc_tcga_pan_can_atlas_2018": "Kidney Clear Cell Carcinoma (KIRC)",
}

GENE_PANEL = ["TP53", "PIK3CA", "PTEN", "EGFR", "KRAS", "BRAF", "MYC", "ERBB2",
              "RB1", "APC", "VHL", "CDKN2A", "ATM", "BRCA1", "BRCA2", "NOTCH1",
              "SMAD4", "STK11", "CTNNB1", "IDH1"]

print(f"Studies to fetch: {len(STUDIES)}")
print(f"Gene panel size: {len(GENE_PANEL)}")


Studies to fetch: 5
Gene panel size: 20


In [3]:
def fetch_entrez_ids(gene_symbols):
    """Map Hugo gene symbols -> Entrez Gene IDs via cBioPortal API."""
    url = f"{BASE_URL}/genes/fetch?geneIdType=HUGO_GENE_SYMBOL&projection=SUMMARY"
    resp = requests.post(url, json=gene_symbols, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return {g["hugoGeneSymbol"]: g["entrezGeneId"] for g in data}

def fetch_clinical_data(study_id):
    """Fetch patient-level clinical data for a study."""
    url = f"{BASE_URL}/studies/{study_id}/clinical-data?clinicalDataType=PATIENT&projection=SUMMARY"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    if df.empty:
        return pd.DataFrame()
    pivot = df.pivot_table(index="patientId", columns="clinicalAttributeId", values="value", aggfunc="first")
    return pivot.reset_index()

def fetch_expression_data(study_id, entrez_map):
    """Fetch mRNA expression Z-scores for the gene panel."""
    profile_id = f"{study_id.replace('_tcga_pan_can_atlas_2018','')}_tcga_pan_can_atlas_2018_rna_seq_v2_mrna_median_Zscores"
    sample_list_id = f"{study_id}_rna_seq_v2_mrna"
    url = f"{BASE_URL}/molecular-profiles/{profile_id}/molecular-data/fetch?projection=SUMMARY"
    body = {"entrezGeneIds": list(entrez_map.values()), "sampleListId": sample_list_id}
    resp = requests.post(url, json=body, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    df = pd.DataFrame(data)
    if df.empty:
        return pd.DataFrame()
    id_to_symbol = {v: k for k, v in entrez_map.items()}
    df["gene"] = df["entrezGeneId"].map(id_to_symbol)
    pivot = df.pivot_table(index="sampleId", columns="gene", values="value", aggfunc="first")
    pivot["patientId"] = pivot.index.str[:12]  # TCGA sample -> patient ID prefix
    return pivot.reset_index(drop=True)


In [4]:
def simulate_fallback_study(study_id, cancer_label, n_patients, gene_panel, seed):
    """Biologically-plausible fallback if the live API call is unavailable."""
    rng = np.random.default_rng(seed)
    driver_bias = {
        "Breast Cancer (BRCA)": ["ERBB2", "PIK3CA", "TP53", "BRCA1", "BRCA2"],
        "Lung Adenocarcinoma (LUAD)": ["EGFR", "KRAS", "STK11", "TP53"],
        "Colorectal Cancer (COADREAD)": ["APC", "KRAS", "TP53", "SMAD4", "BRAF"],
        "Prostate Cancer (PRAD)": ["PTEN", "MYC", "CTNNB1"],
        "Kidney Clear Cell Carcinoma (KIRC)": ["VHL", "PTEN"],
    }[cancer_label]

    rows = []
    for i in range(n_patients):
        row = {"patientId": f"{study_id[:4].upper()}-{i:04d}", "CANCER_TYPE": cancer_label}
        for gene in gene_panel:
            shift = rng.uniform(0.8, 2.2) if gene in driver_bias else 0
            row[gene] = round(rng.normal(0, 1) + (shift if rng.random() < 0.5 else 0), 3)
        row["AGE"] = int(np.clip(rng.normal(61, 11), 25, 90))
        row["SEX"] = rng.choice(["Male", "Female"])
        row["AJCC_PATHOLOGIC_TUMOR_STAGE"] = rng.choice(["Stage I", "Stage II", "Stage III", "Stage IV"], p=[0.3, 0.35, 0.25, 0.10])
        os_months = np.clip(rng.exponential(45), 1, 180)
        row["OS_MONTHS"] = round(os_months, 1)
        event_prob = 0.15 + 0.1 * ["Stage I", "Stage II", "Stage III", "Stage IV"].index(row["AJCC_PATHOLOGIC_TUMOR_STAGE"])
        row["OS_STATUS"] = "1:DECEASED" if rng.random() < event_prob else "0:LIVING"
        rows.append(row)
    return pd.DataFrame(rows)


In [6]:
all_patient_data = []
data_source_log = {}

try:
    entrez_map = fetch_entrez_ids(GENE_PANEL)
    print(f" Resolved {len(entrez_map)} / {len(GENE_PANEL)} Entrez Gene IDs from cBioPortal")
except Exception as e:
    print(f" Could not resolve gene IDs ({e}) — will use simulated fallback for all studies.")
    entrez_map = {}

for study_id, cancer_label in STUDIES.items():
    try:
        if not entrez_map:
            raise RuntimeError("no entrez map")
        clinical = fetch_clinical_data(study_id)
        expression = fetch_expression_data(study_id, entrez_map)
        if clinical.empty or expression.empty:
            raise RuntimeError("empty response")

        merged = pd.merge(clinical, expression, on="patientId", how="inner")
        merged["CANCER_TYPE"] = cancer_label
        all_patient_data.append(merged)
        data_source_log[cancer_label] = f" Live TCGA data ({len(merged)} patients)"
        time.sleep(0.5)  # be polite to the public API
    except Exception as e:
        fallback = simulate_fallback_study(study_id, cancer_label, n_patients=180, gene_panel=GENE_PANEL, seed=hash(study_id) % 10000)
        all_patient_data.append(fallback)
        data_source_log[cancer_label] = f" Simulated fallback ({len(fallback)} patients) — reason: {str(e)[:60]}"

for label, status in data_source_log.items():
    print(f"{label}: {status}")


 Resolved 20 / 20 Entrez Gene IDs from cBioPortal
Breast Cancer (BRCA):  Live TCGA data (1082 patients)
Lung Adenocarcinoma (LUAD):  Live TCGA data (510 patients)
Colorectal Cancer (COADREAD):  Live TCGA data (592 patients)
Prostate Cancer (PRAD):  Live TCGA data (493 patients)
Kidney Clear Cell Carcinoma (KIRC):  Live TCGA data (510 patients)


## 3. Data Integration & Cleaning

In [7]:
pan_cancer_df = pd.concat(all_patient_data, ignore_index=True)

# Standardize columns that may vary slightly between real API and fallback
for col in GENE_PANEL:
    if col not in pan_cancer_df.columns:
        pan_cancer_df[col] = np.nan

pan_cancer_df[GENE_PANEL] = pan_cancer_df[GENE_PANEL].apply(pd.to_numeric, errors="coerce")
pan_cancer_df[GENE_PANEL] = pan_cancer_df[GENE_PANEL].fillna(pan_cancer_df[GENE_PANEL].median())

if "AGE" in pan_cancer_df.columns:
    pan_cancer_df["AGE"] = pd.to_numeric(pan_cancer_df["AGE"], errors="coerce")
    pan_cancer_df["AGE"] = pan_cancer_df["AGE"].fillna(pan_cancer_df["AGE"].median())

if "OS_MONTHS" in pan_cancer_df.columns:
    pan_cancer_df["OS_MONTHS"] = pd.to_numeric(pan_cancer_df["OS_MONTHS"], errors="coerce")
    pan_cancer_df = pan_cancer_df.dropna(subset=["OS_MONTHS"])

pan_cancer_df["event_observed"] = pan_cancer_df["OS_STATUS"].astype(str).str.startswith("1").astype(int)

print(f"Final integrated dataset: {pan_cancer_df.shape[0]} patients x {pan_cancer_df.shape[1]} columns")
print(f"\nPatients per cancer type:\n{pan_cancer_df['CANCER_TYPE'].value_counts()}")
pan_cancer_df[["patientId", "CANCER_TYPE", "AGE", "OS_MONTHS", "OS_STATUS"] + GENE_PANEL[:3]].head()


Final integrated dataset: 3174 patients x 64 columns

Patients per cancer type:
CANCER_TYPE
Breast Cancer (BRCA)                  1082
Colorectal Cancer (COADREAD)           588
Kidney Clear Cell Carcinoma (KIRC)     510
Lung Adenocarcinoma (LUAD)             501
Prostate Cancer (PRAD)                 493
Name: count, dtype: int64


,patientId,CANCER_TYPE,AGE,OS_MONTHS,OS_STATUS,TP53,PIK3CA,PTEN
0,TCGA-3C-AAAU,Breast Cancer (BRCA),55.0,133.050597,0:LIVING,-0.6553,-0.1160,-0.6058
1,TCGA-3C-AALI,Breast Cancer (BRCA),50.0,131.669790,0:LIVING,-1.9283,-0.7887,-1.4641
2,TCGA-3C-AALJ,Breast Cancer (BRCA),62.0,48.459743,0:LIVING,-0.7815,-1.2176,-0.9471
3,TCGA-3C-AALK,Breast Cancer (BRCA),52.0,47.604958,0:LIVING,-0.6123,-0.7720,-0.3935
4,TCGA-4H-AAAK,Breast Cancer (BRCA),50.0,11.440971,0:LIVING,-0.5259,-0.4289,-0.5846


## 4. Exploratory Data Analysis

In [8]:
fig = px.bar(pan_cancer_df["CANCER_TYPE"].value_counts(), orientation='h',
             title="Patient Count per Cancer Type", template="plotly_white",
             labels={"value": "Patients", "index": "Cancer Type"},
             color=pan_cancer_df["CANCER_TYPE"].value_counts().values, color_continuous_scale="Tealgrn")
fig.update_layout(height=400, showlegend=False)
fig.show()

fig2 = px.histogram(pan_cancer_df, x="AGE", color="CANCER_TYPE", barmode="overlay", nbins=30,
                     title="Age Distribution Across Cancer Types", template="plotly_white", opacity=0.6)
fig2.update_layout(height=450)
fig2.show()


In [9]:
fig = px.box(pan_cancer_df, x="CANCER_TYPE", y="OS_MONTHS", color="CANCER_TYPE",
             title="Overall Survival Time Distribution by Cancer Type", template="plotly_white")
fig.update_layout(height=450, showlegend=False)
fig.show()

event_by_cancer = pan_cancer_df.groupby("CANCER_TYPE")["event_observed"].mean().sort_values(ascending=False)
fig2 = px.bar(event_by_cancer, title="Mortality Rate by Cancer Type (in this cohort)",
              labels={"value": "Fraction Deceased", "index": "Cancer Type"}, template="plotly_white",
              color=event_by_cancer.values, color_continuous_scale="Reds")
fig2.update_layout(height=400, showlegend=False)
fig2.show()


In [11]:
expr_by_cancer = pan_cancer_df.groupby("CANCER_TYPE")[GENE_PANEL].mean()

fig = px.imshow(expr_by_cancer.values, x=GENE_PANEL, y=expr_by_cancer.index,
                 color_continuous_scale="RdBu_r", color_continuous_midpoint=0, aspect="auto",
                 title="Average Gene Expression (Z-score) by Cancer Type",
                 labels=dict(color="Mean Z-score"))
fig.update_layout(height=450, xaxis_tickangle=-45)
fig.show()

## 5. Pan-Cancer Type Classification

In [12]:
le = LabelEncoder()
X = pan_cancer_df[GENE_PANEL]
y = le.fit_transform(pan_cancer_df["CANCER_TYPE"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

pancancer_scaler = StandardScaler()
X_train_scaled = pancancer_scaler.fit_transform(X_train)
X_test_scaled = pancancer_scaler.transform(X_test)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

model_results = []
trained = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained[name] = model
    preds = model.predict(X_test_scaled)
    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Macro F1": f1_score(y_test, preds, average="macro")
    })

results_df = pd.DataFrame(model_results).sort_values("Accuracy", ascending=False)
results_df


,Model,Accuracy,Macro F1
0,Random Forest,0.761965,0.736301
1,Gradient Boosting,0.753149,0.723506


In [13]:
best_model_name = results_df.iloc[0]["Model"]
pancancer_clf = trained[best_model_name]
print(f"Best model: {best_model_name}")

preds = pancancer_clf.predict(X_test_scaled)
print(classification_report(y_test, preds, target_names=le.classes_))


Best model: Random Forest
                                    precision    recall  f1-score   support

              Breast Cancer (BRCA)       0.72      0.99      0.84       271
      Colorectal Cancer (COADREAD)       0.81      0.63      0.71       147
Kidney Clear Cell Carcinoma (KIRC)       0.82      0.66      0.73       128
        Lung Adenocarcinoma (LUAD)       0.75      0.65      0.70       125
            Prostate Cancer (PRAD)       0.81      0.63      0.71       123

                          accuracy                           0.76       794
                         macro avg       0.78      0.71      0.74       794
                      weighted avg       0.77      0.76      0.75       794



## 6. Model Evaluation — Confusion Matrix & ROC (One-vs-Rest)

In [14]:
cm = confusion_matrix(y_test, preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                 x=le.classes_, y=le.classes_,
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 title=f"Confusion Matrix — {best_model_name} (Pan-Cancer Classifier)")
fig.update_layout(height=550, xaxis_tickangle=-30)
fig.show()


In [15]:
y_test_bin = label_binarize(y_test, classes=np.arange(len(le.classes_)))
probs = pancancer_clf.predict_proba(X_test_scaled)

fig = go.Figure()
for i, cls in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], probs[:, i])
    auc = roc_auc_score(y_test_bin[:, i], probs[:, i])
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f"{cls} (AUC={auc:.3f})"))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random', line=dict(dash='dash', color='gray')))
fig.update_layout(title="ROC Curves — One-vs-Rest (per Cancer Type)", xaxis_title="False Positive Rate",
                   yaxis_title="True Positive Rate", template="plotly_white", height=550)
fig.show()

importances = pd.Series(pancancer_clf.feature_importances_, index=GENE_PANEL).sort_values(ascending=False)
fig2 = px.bar(importances, orientation='h', title="Gene Importance for Cancer Type Classification",
              template="plotly_white", labels={"value": "Importance", "index": "Gene"},
              color=importances.values, color_continuous_scale="Viridis")
fig2.update_layout(height=500, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()


## 7. Kaplan-Meier Survival Analysis

In [16]:
kmf = KaplanMeierFitter()

fig = go.Figure()
colors_km = px.colors.qualitative.Set2

for i, cancer_type in enumerate(pan_cancer_df["CANCER_TYPE"].unique()):
    subset = pan_cancer_df[pan_cancer_df["CANCER_TYPE"] == cancer_type]
    kmf.fit(subset["OS_MONTHS"], subset["event_observed"], label=cancer_type)
    surv = kmf.survival_function_
    fig.add_trace(go.Scatter(x=surv.index, y=surv[cancer_type], mode='lines', name=cancer_type,
                              line=dict(color=colors_km[i % len(colors_km)], width=2)))

fig.update_layout(title="Kaplan-Meier Overall Survival by Cancer Type", xaxis_title="Months",
                   yaxis_title="Survival Probability", template="plotly_white", height=550)
fig.show()

# Log-rank test across all cancer types
result = multivariate_logrank_test(pan_cancer_df["OS_MONTHS"], pan_cancer_df["CANCER_TYPE"], pan_cancer_df["event_observed"])
print(f"Multivariate log-rank test p-value: {result.p_value:.2e}")
print("(p < 0.05 => survival significantly differs across cancer types)")


Multivariate log-rank test p-value: 1.51e-56
(p < 0.05 => survival significantly differs across cancer types)


## 8. Gene-Stratified Survival — High vs Low Expression

In [17]:
def plot_gene_survival(gene, df):
    median_expr = df[gene].median()
    high_group = df[df[gene] > median_expr]
    low_group = df[df[gene] <= median_expr]

    kmf_high, kmf_low = KaplanMeierFitter(), KaplanMeierFitter()
    kmf_high.fit(high_group["OS_MONTHS"], high_group["event_observed"], label=f"{gene} High")
    kmf_low.fit(low_group["OS_MONTHS"], low_group["event_observed"], label=f"{gene} Low")

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=kmf_high.survival_function_.index, y=kmf_high.survival_function_[f"{gene} High"],
                              mode='lines', name=f"{gene} High Expression", line=dict(color="#E63946", width=2)))
    fig.add_trace(go.Scatter(x=kmf_low.survival_function_.index, y=kmf_low.survival_function_[f"{gene} Low"],
                              mode='lines', name=f"{gene} Low Expression", line=dict(color="#2E86AB", width=2)))

    lr = logrank_test(high_group["OS_MONTHS"], low_group["OS_MONTHS"],
                       high_group["event_observed"], low_group["event_observed"])
    fig.update_layout(title=f"Survival by {gene} Expression (log-rank p={lr.p_value:.4f})",
                       xaxis_title="Months", yaxis_title="Survival Probability", template="plotly_white", height=450)
    fig.show()
    return lr.p_value

# Check the top 3 most important genes from the classifier for survival association
top_genes = importances.head(3).index.tolist()
gene_pvals = {}
for gene in top_genes:
    gene_pvals[gene] = plot_gene_survival(gene, pan_cancer_df)

print("\nLog-rank p-values (top classifier genes):")
for g, p in gene_pvals.items():
    sig = "significant " if p < 0.05 else "not significant"
    print(f"  {g}: p={p:.4f} ({sig})")



Log-rank p-values (top classifier genes):
  CDKN2A: p=0.0475 (significant ✅)
  EGFR: p=0.7169 (not significant)
  RB1: p=0.5650 (not significant)


## 9. Cox Proportional Hazards Regression

In [18]:
cox_df = pan_cancer_df[["OS_MONTHS", "event_observed", "AGE"] + GENE_PANEL].copy()
cox_df = cox_df.dropna()

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col="OS_MONTHS", event_col="event_observed")

cox_summary = cph.summary[["coef", "exp(coef)", "se(coef)", "p"]].sort_values("p")
cox_summary.columns = ["Coefficient", "Hazard_Ratio", "Std_Error", "p_value"]
cox_summary


,Coefficient,Hazard_Ratio,Std_Error,p_value
covariate,,,,
AGE,0.023404,1.023680,0.002744,1.461783e-17
CDKN2A,0.050156,1.051435,0.015037,8.514410e-04
PIK3CA,0.025041,1.025357,0.008477,3.137574e-03
CTNNB1,-0.074901,0.927835,0.030814,1.506865e-02
KRAS,0.038369,1.039114,0.017273,2.632703e-02
TP53,0.055616,1.057191,0.028053,4.742281e-02
BRCA1,0.052001,1.053377,0.027823,6.162442e-02
APC,-0.060458,0.941334,0.033550,7.154494e-02
IDH1,0.042193,1.043096,0.027299,1.222055e-01


In [19]:
# Forest plot of hazard ratios
forest_df = cox_summary.reset_index().rename(columns={"index": "Feature", "covariate": "Feature"})
forest_df["significant"] = forest_df["p_value"] < 0.05
forest_df["ci_lower"] = np.exp(cox_summary["Coefficient"] - 1.96 * cox_summary["Std_Error"]).values
forest_df["ci_upper"] = np.exp(cox_summary["Coefficient"] + 1.96 * cox_summary["Std_Error"]).values
forest_df = forest_df.sort_values("Hazard_Ratio")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=forest_df["Hazard_Ratio"], y=forest_df["Feature"], mode='markers',
    marker=dict(size=10, color=np.where(forest_df["significant"], "#E63946", "#999")),
    error_x=dict(type='data', symmetric=False,
                 array=forest_df["ci_upper"] - forest_df["Hazard_Ratio"],
                 arrayminus=forest_df["Hazard_Ratio"] - forest_df["ci_lower"])
))
fig.add_vline(x=1, line_dash="dash", line_color="gray", annotation_text="HR=1 (no effect)")
fig.update_layout(title="Cox Regression — Hazard Ratios (Forest Plot, 95% CI)",
                   xaxis_title="Hazard Ratio", template="plotly_white", height=600)
fig.show()

print(" HR > 1 = gene/factor associated with HIGHER risk of death")
print(" HR < 1 = gene/factor associated with LOWER risk (protective)")


💡 HR > 1 = gene/factor associated with HIGHER risk of death
💡 HR < 1 = gene/factor associated with LOWER risk (protective)


## 10. Multivariate Risk Score & Patient Stratification

In [20]:
cox_df["risk_score"] = cph.predict_partial_hazard(cox_df)
median_risk = cox_df["risk_score"].median()
cox_df["risk_group"] = np.where(cox_df["risk_score"] > median_risk, "High Risk", "Low Risk")

kmf_h, kmf_l = KaplanMeierFitter(), KaplanMeierFitter()
high_r = cox_df[cox_df["risk_group"] == "High Risk"]
low_r = cox_df[cox_df["risk_group"] == "Low Risk"]
kmf_h.fit(high_r["OS_MONTHS"], high_r["event_observed"], label="High Risk")
kmf_l.fit(low_r["OS_MONTHS"], low_r["event_observed"], label="Low Risk")

fig = go.Figure()
fig.add_trace(go.Scatter(x=kmf_h.survival_function_.index, y=kmf_h.survival_function_["High Risk"],
                          mode='lines', name="High Risk (Cox score)", line=dict(color="#E63946", width=3)))
fig.add_trace(go.Scatter(x=kmf_l.survival_function_.index, y=kmf_l.survival_function_["Low Risk"],
                          mode='lines', name="Low Risk (Cox score)", line=dict(color="#43AA8B", width=3)))

lr_risk = logrank_test(high_r["OS_MONTHS"], low_r["OS_MONTHS"], high_r["event_observed"], low_r["event_observed"])
fig.update_layout(title=f"Multivariate Risk Stratification (log-rank p={lr_risk.p_value:.2e})",
                   xaxis_title="Months", yaxis_title="Survival Probability", template="plotly_white", height=500)
fig.show()

print(f"High Risk group: {len(high_r)} patients | Low Risk group: {len(low_r)} patients")
print(f"Log-rank p-value: {lr_risk.p_value:.2e} — {'significant separation ' if lr_risk.p_value < 0.05 else 'not significant'}")


High Risk group: 1587 patients | Low Risk group: 1587 patients
Log-rank p-value: 1.02e-34 — significant separation ✅


## 11.  Runtime Prediction — Apna Patient Data Direct Input Karein

Neeche patient ki **age**, **20 gene expression Z-scores**, aur clinical details daalein — system do cheezein predict karega:

1. **Predicted Cancer Type** (Pan-Cancer classifier se)
2. **Risk Group** (Cox regression model se) — High Risk ya Low Risk, saath mein estimated survival curve


In [21]:
age_input = widgets.IntSlider(value=60, min=20, max=90, description="Age", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))

gene_defaults_pc = pan_cancer_df[GENE_PANEL].mean().to_dict()
gene_boxes_pc = {}
gene_widgets_pc = []
for gene in GENE_PANEL:
    box = widgets.FloatText(value=round(gene_defaults_pc[gene], 2), description=gene,
                             style={'description_width': '90px'}, layout=widgets.Layout(width='220px'))
    gene_boxes_pc[gene] = box
    gene_widgets_pc.append(box)

gene_grid_pc = widgets.GridBox(gene_widgets_pc, layout=widgets.Layout(grid_template_columns="repeat(5, 230px)", grid_gap="6px"))

predict_btn = widgets.Button(description=" Analyze Patient", button_style='success',
                              layout=widgets.Layout(width='250px', height='42px'))
out = widgets.Output()

def render_pancancer_result(cancer_pred, cancer_proba, classes, risk_group, risk_score, survival_curve):
    top3_idx = cancer_proba.argsort()[::-1][:3]
    top3_html = "".join(
        f'<div style="display:flex; justify-content:space-between; font-size:13px; margin-top:4px;">'
        f'<span>{classes[i]}</span><span><b>{cancer_proba[i]*100:.1f}%</b></span></div>'
        for i in top3_idx
    )

    risk_color = "#E63946" if risk_group == "High Risk" else "#43AA8B"
    risk_emoji = "" if risk_group == "High Risk" else ""

    surv_x = survival_curve.index.tolist()
    surv_y = survival_curve.iloc[:, 0].tolist()

    html = f"""
    <div style="font-family:sans-serif;">
        <div style="border:2px solid #2C5364; border-radius:12px; padding:18px; background:#fafafa;">
            <div style="font-size:19px; font-weight:700; color:#2C5364;"> Predicted Cancer Type: {classes[top3_idx[0]]}</div>
            <div style="font-size:13px; margin-top:8px; color:#333;"><b>Top 3 Probabilities:</b></div>
            {top3_html}
        </div>
        <div style="border:2px solid {risk_color}; border-radius:12px; padding:18px; margin-top:14px; background:#fafafa;">
            <div style="font-size:19px; font-weight:700; color:{risk_color};">{risk_emoji} Survival Risk Group: {risk_group}</div>
            <div style="font-size:13px; margin-top:6px; color:#333;">Cox Risk Score: <b>{risk_score:.3f}</b></div>
        </div>
    </div>
    """
    display(HTML(html))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=surv_x, y=surv_y, mode='lines', line=dict(color=risk_color, width=3),
                              name=f"Estimated Survival ({risk_group})"))
    fig.update_layout(title="Estimated Survival Curve for This Risk Profile", xaxis_title="Months",
                       yaxis_title="Survival Probability", template="plotly_white", height=400)
    fig.show()

def on_predict(b):
    with out:
        clear_output()
        gene_vals = {g: gene_boxes_pc[g].value for g in GENE_PANEL}

        # Cancer type prediction
        row_df = pd.DataFrame([gene_vals])[GENE_PANEL]
        row_scaled = pancancer_scaler.transform(row_df)
        cancer_proba = pancancer_clf.predict_proba(row_scaled)[0]

        # Cox risk score
        cox_row = pd.DataFrame([{**gene_vals, "AGE": age_input.value}])[["AGE"] + GENE_PANEL]
        risk_score = cph.predict_partial_hazard(cox_row).values[0]
        risk_group = "High Risk" if risk_score > median_risk else "Low Risk"

        baseline_surv = cph.predict_survival_function(cox_row)

        render_pancancer_result(pancancer_clf.predict(row_scaled)[0], cancer_proba, le.classes_,
                                 risk_group, risk_score, baseline_surv)

predict_btn.on_click(on_predict)

display(age_input)
display(widgets.HTML("<br><b style='font-size:15px;'>Gene Expression Panel (Z-scores)</b>"))
display(gene_grid_pc)
display(predict_btn)
display(out)


IntSlider(value=60, description='Age', layout=Layout(width='400px'), max=90, min=20, style=SliderStyle(descrip…

HTML(value="<br><b style='font-size:15px;'>Gene Expression Panel (Z-scores)</b>")

GridBox(children=(FloatText(value=-0.29, description='TP53', layout=Layout(width='220px'), style=DescriptionSt…

Button(button_style='success', description='🧬 Analyze Patient', layout=Layout(height='42px', width='250px'), s…

Output()